## Air Pollution (PM₂.₅) & County-level Health Outcomes in U.S.

## Dataset(s) to be used  
- Air quality dataset (county level): "/Users/huangzhihan/Desktop/zh2695-debug.github.io/air_quality.csv"
- Health outcomes dataset (county level): "/Users/huangzhihan/Desktop/zh2695-debug.github.io/Ranked_Measure_Data_2023.csv" 

## Analysis question  
Across U.S. counties, is there a correlation (or association) between county-level：air pollution (PM₂.₅ exposure) and health outcomes

## Columns that will (likely) be used  
- From air quality dataset: `PM2.5 Wtd AM (µg/m3)` (renamed to `pm25_wtd`), `PM2.5 24-hr (µg/m3)` (renamed to `pm25_24h`)  
- From health dataset: `Poor Physical Health Days`, `Poor Mental Health Days`  
- Geographic / merge keys: `County` (and optionally `State`)  

## Columns used to merge/join datasets  
- Air quality dataset: `County`  
- Health dataset: `County`  

## Hypothesis  
- **H1**: Higher county-level PM₂.₅ exposure (as measured by composite of weighted average and 24-hr) is associated with *higher* average number of physically unhealthy days.  
- **H2**: Higher county-level PM₂.₅ exposure is associated with *higher* average number of mentally unhealthy days.  

## Firstly, I import necessary packages.

In [249]:
import os

import pandas as pd

import plotly.express as px

## Because my data is excel, I convert two dataset into csv format.

In [250]:
xlsx_path = "/Users/huangzhihan/Desktop/zh2695-debug.github.io/County Health Data.xlsx"

csv_path = (
    "/Users/huangzhihan/Desktop/zh2695-debug.github.io/Ranked_Measure_Data_2023.csv"
)

df1 = pd.read_excel(xlsx_path, sheet_name="Ranked Measure Data")

df1.to_csv(csv_path, index=False, encoding="utf-8")


In [251]:
xlsx_path = "/Users/huangzhihan/Desktop/zh2695-debug.github.io/ctyfactbook2023.xlsx"

csv_path = "/Users/huangzhihan/Desktop/zh2695-debug.github.io/air_quality.csv"

df2 = pd.read_excel(xlsx_path)

df2.to_csv(csv_path, index=False, encoding="utf-8")

## Second, I begin to process the first dataframe. 

In [252]:
df1 = pd.read_csv("Ranked_Measure_Data_2023.csv")

In [253]:
df1.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Premature Death,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Poor or Fair Health,Unnamed: 30,Unnamed: 31,Unnamed: 32,Poor Physical Health Days,Unnamed: 34,Unnamed: 35,Unnamed: 36,Poor Mental Health Days,Unnamed: 38,Unnamed: 39,Unnamed: 40,Low Birthweight,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56,Unnamed: 57,Unnamed: 58,Unnamed: 59,Unnamed: 60,Adult Smoking,Unnamed: 62,Unnamed: 63,Unnamed: 64,Adult Obesity,Unnamed: 66,Unnamed: 67,Unnamed: 68,Food Environment Index,Unnamed: 70,Physical Inactivity,Unnamed: 72,Unnamed: 73,Unnamed: 74,Access to Exercise Opportunities,Unnamed: 76,Excessive Drinking,Unnamed: 78,Unnamed: 79,Unnamed: 80,Alcohol-Impaired Driving Deaths,Unnamed: 82,Unnamed: 83,Unnamed: 84,Unnamed: 85,Unnamed: 86,Sexually Transmitted Infections,Unnamed: 88,Unnamed: 89,Teen Births,Unnamed: 91,Unnamed: 92,Unnamed: 93,Unnamed: 94,Unnamed: 95,Unnamed: 96,Unnamed: 97,Unnamed: 98,Unnamed: 99,Unnamed: 100,Unnamed: 101,Unnamed: 102,Unnamed: 103,Unnamed: 104,Unnamed: 105,Unnamed: 106,Unnamed: 107,Unnamed: 108,Uninsured,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Primary Care Physicians,Unnamed: 115,Unnamed: 116,Unnamed: 117,Dentists,Unnamed: 119,Unnamed: 120,Unnamed: 121,Mental Health Providers,Unnamed: 123,Unnamed: 124,Unnamed: 125,Preventable Hospital Stays,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Mammography Screening,Unnamed: 134,Unnamed: 135,Unnamed: 136,Unnamed: 137,Unnamed: 138,Unnamed: 139,Flu Vaccinations,Unnamed: 141,Unnamed: 142,Unnamed: 143,Unnamed: 144,Unnamed: 145,Unnamed: 146,High School Completion,Unnamed: 148,Unnamed: 149,Unnamed: 150,Unnamed: 151,Unnamed: 152,Some College,Unnamed: 154,Unnamed: 155,Unnamed: 156,Unnamed: 157,Unnamed: 158,Unemployment,Unnamed: 160,Unnamed: 161,Unnamed: 162,Children in Poverty,Unnamed: 164,Unnamed: 165,Unnamed: 166,Unnamed: 167,Unnamed: 168,Unnamed: 169,Unnamed: 170,Unnamed: 171,Income Inequality,Unnamed: 173,Unnamed: 174,Unnamed: 175,Children in Single-Parent Households,Unnamed: 177,Unnamed: 178,Unnamed: 179,Unnamed: 180,Unnamed: 181,Social Associations,Unnamed: 183,Unnamed: 184,Injury Deaths,Unnamed: 186,Unnamed: 187,Unnamed: 188,Unnamed: 189,Unnamed: 190,Unnamed: 191,Unnamed: 192,Unnamed: 193,Unnamed: 194,Unnamed: 195,Unnamed: 196,Unnamed: 197,Unnamed: 198,Unnamed: 199,Unnamed: 200,Unnamed: 201,Unnamed: 202,Unnamed: 203,Unnamed: 204,Air Pollution - Particulate Matter,Unnamed: 206,Drinking Water Violations,Unnamed: 208,Severe Housing Problems,Unnamed: 210,Unnamed: 211,Unnamed: 212,Unnamed: 213,Unnamed: 214,Unnamed: 215,Unnamed: 216,Unnamed: 217,Unnamed: 218,Unnamed: 219,Unnamed: 220,Unnamed: 221,Driving Alone to Work,Unnamed: 223,Unnamed: 224,Unnamed: 225,Unnamed: 226,Unnamed: 227,Unnamed: 228,Unnamed: 229,Unnamed: 230,Unnamed: 231,Unnamed: 232,Unnamed: 233,Unnamed: 234,Unnamed: 235,Unnamed: 236,Unnamed: 237,Unnamed: 238,Unnamed: 239,Unnamed: 240,Long Commute - Driving Alone,Unnamed: 242,Unnamed: 243,Unnamed: 244,Unnamed: 245
0,FIPS,State,County,Unreliable,Deaths,Years of Potential Life Lost Rate,95% CI - Low,95% CI - High,Quartile,YPLL Rate (AIAN),YPLL Rate (AIAN) 95% CI - Low,YPLL Rate (AIAN) 95% CI - High,YPLL Rate (AIAN) Unreliable,YPLL Rate (Asian),YPLL Rate (Asian) 95% CI - Low,YPLL Rate (Asian) 95% CI - High,YPLL Rate (Asian) Unreliable,YPLL Rate (Black),YPLL Rate (Black) 95% CI - Low,YPLL Rate (Black) 95% CI - High,YPLL Rate (Black) Unreliable,YPLL Rate (Hispanic),YPLL Rate (Hispanic) 95% CI - Low,YPLL Rate (Hispanic) 95% CI - High,YPLL Rate (Hispanic) Unreliable,YPLL Rate (White),YPLL Rate (White) 95% CI - Low,YPLL 

In [254]:
df1 = df1.rename(columns={"Unnamed: 1": "State", "Unnamed: 2": "County"})


In [255]:
df1.head()

,Unnamed: 0,State,County,Premature Death,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Poor or Fair Health,Unnamed: 30,Unnamed: 31,Unnamed: 32,Poor Physical Health Days,Unnamed: 34,Unnamed: 35,Unnamed: 36,Poor Mental Health Days,Unnamed: 38,Unnamed: 39,Unnamed: 40,Low Birthweight,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56,Unnamed: 57,Unnamed: 58,Unnamed: 59,Unnamed: 60,Adult Smoking,Unnamed: 62,Unnamed: 63,Unnamed: 64,Adult Obesity,Unnamed: 66,Unnamed: 67,Unnamed: 68,Food Environment Index,Unnamed: 70,Physical Inactivity,Unnamed: 72,Unnamed: 73,Unnamed: 74,Access to Exercise Opportunities,Unnamed: 76,Excessive Drinking,Unnamed: 78,Unnamed: 79,Unnamed: 80,Alcohol-Impaired Driving Deaths,Unnamed: 82,Unnamed: 83,Unnamed: 84,Unnamed: 85,Unnamed: 86,Sexually Transmitted Infections,Unnamed: 88,Unnamed: 89,Teen Births,Unnamed: 91,Unnamed: 92,Unnamed: 93,Unnamed: 94,Unnamed: 95,Unnamed: 96,Unnamed: 97,Unnamed: 98,Unnamed: 99,Unnamed: 100,Unnamed: 101,Unnamed: 102,Unnamed: 103,Unnamed: 104,Unnamed: 105,Unnamed: 106,Unnamed: 107,Unnamed: 108,Uninsured,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Primary Care Physicians,Unnamed: 115,Unnamed: 116,Unnamed: 117,Dentists,Unnamed: 119,Unnamed: 120,Unnamed: 121,Mental Health Providers,Unnamed: 123,Unnamed: 124,Unnamed: 125,Preventable Hospital Stays,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Mammography Screening,Unnamed: 134,Unnamed: 135,Unnamed: 136,Unnamed: 137,Unnamed: 138,Unnamed: 139,Flu Vaccinations,Unnamed: 141,Unnamed: 142,Unnamed: 143,Unnamed: 144,Unnamed: 145,Unnamed: 146,High School Completion,Unnamed: 148,Unnamed: 149,Unnamed: 150,Unnamed: 151,Unnamed: 152,Some College,Unnamed: 154,Unnamed: 155,Unnamed: 156,Unnamed: 157,Unnamed: 158,Unemployment,Unnamed: 160,Unnamed: 161,Unnamed: 162,Children in Poverty,Unnamed: 164,Unnamed: 165,Unnamed: 166,Unnamed: 167,Unnamed: 168,Unnamed: 169,Unnamed: 170,Unnamed: 171,Income Inequality,Unnamed: 173,Unnamed: 174,Unnamed: 175,Children in Single-Parent Households,Unnamed: 177,Unnamed: 178,Unnamed: 179,Unnamed: 180,Unnamed: 181,Social Associations,Unnamed: 183,Unnamed: 184,Injury Deaths,Unnamed: 186,Unnamed: 187,Unnamed: 188,Unnamed: 189,Unnamed: 190,Unnamed: 191,Unnamed: 192,Unnamed: 193,Unnamed: 194,Unnamed: 195,Unnamed: 196,Unnamed: 197,Unnamed: 198,Unnamed: 199,Unnamed: 200,Unnamed: 201,Unnamed: 202,Unnamed: 203,Unnamed: 204,Air Pollution - Particulate Matter,Unnamed: 206,Drinking Water Violations,Unnamed: 208,Severe Housing Problems,Unnamed: 210,Unnamed: 211,Unnamed: 212,Unnamed: 213,Unnamed: 214,Unnamed: 215,Unnamed: 216,Unnamed: 217,Unnamed: 218,Unnamed: 219,Unnamed: 220,Unnamed: 221,Driving Alone to Work,Unnamed: 223,Unnamed: 224,Unnamed: 225,Unnamed: 226,Unnamed: 227,Unnamed: 228,Unnamed: 229,Unnamed: 230,Unnamed: 231,Unnamed: 232,Unnamed: 233,Unnamed: 234,Unnamed: 235,Unnamed: 236,Unnamed: 237,Unnamed: 238,Unnamed: 239,Unnamed: 240,Long Commute - Driving Alone,Unnamed: 242,Unnamed: 243,Unnamed: 244,Unnamed: 245
0,FIPS,State,County,Unreliable,Deaths,Years of Potential Life Lost Rate,95% CI - Low,95% CI - High,Quartile,YPLL Rate (AIAN),YPLL Rate (AIAN) 95% CI - Low,YPLL Rate (AIAN) 95% CI - High,YPLL Rate (AIAN) Unreliable,YPLL Rate (Asian),YPLL Rate (Asian) 95% CI - Low,YPLL Rate (Asian) 95% CI - High,YPLL Rate (Asian) Unreliable,YPLL Rate (Black),YPLL Rate (Black) 95% CI - Low,YPLL Rate (Black) 95% CI - High,YPLL Rate (Black) Unreliable,YPLL Rate (Hispanic),YPLL Rate (Hispanic) 95% CI - Low,YPLL Rate (Hispanic) 95% CI - High,YPLL Rate (Hispanic) Unreliable,YPLL Rate (White),YPLL Rate (White) 95% CI - Low,YPLL Rate (Whi

## I extracted the useful columns from dataframe.

In [256]:
df1 = df1[
    [
        "State",
        "County",
        "Air Pollution - Particulate Matter",
        "Premature Death",
        "Poor Physical Health Days",
        "Poor Mental Health Days",
    ]
]

In [257]:
df1.head()

,State,County,Air Pollution - Particulate Matter,Premature Death,Poor Physical Health Days,Poor Mental Health Days
0,State,County,Average Daily PM2.5,Unreliable,Average Number of Physically Unhealthy Days,Average Number of Mentally Unhealthy Days
1,Alabama,NaN,9.3,NaN,3.4824161407,5.0732772786
2,Alabama,Autauga,10,NaN,3.4322111963,4.7973512632
3,Alabama,Baldwin,7.6,NaN,3.2761769521,4.7537497106
4,Alabama,Barbour,9.4,NaN,4.6054318631,4.9548545646


In [258]:
df1.drop("Premature Death", axis=1, inplace=True)

In [259]:
df1.head()

,State,County,Air Pollution - Particulate Matter,Poor Physical Health Days,Poor Mental Health Days
0,State,County,Average Daily PM2.5,Average Number of Physically Unhealthy Days,Average Number of Mentally Unhealthy Days
1,Alabama,NaN,9.3,3.4824161407,5.0732772786
2,Alabama,Autauga,10,3.4322111963,4.7973512632
3,Alabama,Baldwin,7.6,3.2761769521,4.7537497106
4,Alabama,Barbour,9.4,4.6054318631,4.9548545646


## Third, I processed the second dataframe.

In [260]:
df2 = pd.read_csv("air_quality.csv")
df2.columns

Index(['State', 'County', 'County FIPS Code', '2010 Population',
       'CO          8-hr (ppm)', 'Pb           3-mo (µg/m3)',
       'NO2         AM (ppb)', 'NO2          1-hr (ppb)',
       'O3            8-hr (ppm)', 'PM10        24-hr (µg/m3) ',
       'PM2.5     Wtd AM (µg/m3) ', 'PM2.5     24-hr (µg/m3) ',
       'SO2         1-hr (ppb)'],
      dtype='object')

## Because the original columns' names are so complicated, I rename them to make them read easier.

In [261]:
df2.rename(
    columns={
        "PM2.5     Wtd AM (µg/m3) ": "pm25_wtd",
        "PM2.5     24-hr (µg/m3) ": "pm25_24h",
    },
    inplace=True,
)
df2.head()

,State,County,County FIPS Code,2010 Population,CO 8-hr (ppm),Pb 3-mo (µg/m3),NO2 AM (ppb),NO2 1-hr (ppb),O3 8-hr (ppm),PM10 24-hr (µg/m3),pm25_wtd,pm25_24h,SO2 1-hr (ppb)
0,Alabama,Baldwin County,1003.0,182265.0,ND,ND,ND,ND,0.065,ND,7.6,18,ND
1,Alabama,Clay County,1027.0,13932.0,ND,ND,ND,ND,ND,ND,IN,IN,ND
2,Alabama,Colbert County,1033.0,54428.0,ND,ND,ND,ND,ND,ND,IN,IN,ND
3,Alabama,DeKalb County,1049.0,71109.0,ND,ND,ND,ND,0.066,ND,8.9,21,ND
4,Alabama,Elmore County,1051.0,79303.0,ND,ND,ND,ND,0.061,ND,ND,ND,ND


## To assign weights to the two different PM₂.₅ metrics.
Prior epidemiological studies suggest that both chronic exposure to fine particulate matter (PM₂.₅) and short-term high-concentration events may contribute to adverse health outcomes such as respiratory illness, cardiovascular stress, and reduced general well-being.  
Because my analysis examines county-level health metrics , I believe that a composite exposure measure that combines long-term pollution burden and potential acute spikes better reflects residents’ exposure than either metric alone.  
Thus, I construct a composite PM₂.₅ measure as:
new_pm2.5 = 0.7 * annual_avg_PM2.5 + 0.3 * daily_24h_PM2.5

In [262]:
df2["pm25_wtd"] = pd.to_numeric(df2["pm25_wtd"], errors="coerce")
df2["pm25_24h"] = pd.to_numeric(df2["pm25_24h"], errors="coerce")

w1 = 0.7
w2 = 0.3

df2["new_pm2.5"] = df2["pm25_wtd"] * w1 + w2 * df2["pm25_24h"]
df2.head()


,State,County,County FIPS Code,2010 Population,CO 8-hr (ppm),Pb 3-mo (µg/m3),NO2 AM (ppb),NO2 1-hr (ppb),O3 8-hr (ppm),PM10 24-hr (µg/m3),pm25_wtd,pm25_24h,SO2 1-hr (ppb),new_pm2.5
0,Alabama,Baldwin County,1003.0,182265.0,ND,ND,ND,ND,0.065,ND,7.6,18.0,ND,10.72
1,Alabama,Clay County,1027.0,13932.0,ND,ND,ND,ND,ND,ND,NaN,NaN,ND,NaN
2,Alabama,Colbert County,1033.0,54428.0,ND,ND,ND,ND,ND,ND,NaN,NaN,ND,NaN
3,Alabama,DeKalb County,1049.0,71109.0,ND,ND,ND,ND,0.066,ND,8.9,21.0,ND,12.53
4,Alabama,Elmore County,1051.0,79303.0,ND,ND,ND,ND,0.061,ND,NaN,NaN,ND,NaN


## I keep the necessary columns

In [263]:
dfpm25 = df2[["State", "County", "new_pm2.5"]]
dfpm25.head()

,State,County,new_pm2.5
0,Alabama,Baldwin County,10.72
1,Alabama,Clay County,NaN
2,Alabama,Colbert County,NaN
3,Alabama,DeKalb County,12.53
4,Alabama,Elmore County,NaN


## I merged two dataframe on "Country", so that I can process the one dataframe and draw a chart.

In [264]:
newdf = pd.merge(df1, dfpm25, on="County", how="outer")
newdf.head()

,State_x,County,Air Pollution - Particulate Matter,Poor Physical Health Days,Poor Mental Health Days,State_y,new_pm2.5
0,South Carolina,Abbeville,9.2,3.990457608,5.0897933139,NaN,NaN
1,Louisiana,Acadia,8.6,4.3123100071,6.1451431097,NaN,NaN
2,Virginia,Accomack,6.9,3.9489105646,4.8148290762,NaN,NaN
3,Idaho,Ada,5.4,2.8275698146,4.2961462333,NaN,NaN
4,NaN,Ada County,NaN,NaN,NaN,Idaho,NaN


In [265]:
print(newdf.columns.tolist())

['State_x', 'County', 'Air Pollution - Particulate Matter', 'Poor Physical Health Days', 'Poor Mental Health Days', 'State_y', 'new_pm2.5']


## Because I need to draw scatter plot, first I should transfer all data to numerical value.

In [ ]:
newdf["Poor Physical Health Days"] = pd.to_numeric(
    newdf["Poor Physical Health Days"], errors="coerce"
)
newdf["Poor Mental Health Days"] = pd.to_numeric(
    newdf["Poor Mental Health Days"], errors="coerce"
)


newdf["new_pm2.5"] = pd.to_numeric(newdf["new_pm2.5"], errors="coerce")


newdf = newdf.dropna(
    subset=["new_pm2.5", "Poor Physical Health Days", "Poor Mental Health Days"]
)


## Finally I use plotly package to draw scatter plot.

In [ ]:
fig1 = px.scatter(
    newdf,
    x="new_pm2.5",
    y="Poor Physical Health Days",
    trendline="ols",
    labels={
        "pm25_composite": "PM2.5 composite (µg/m³)",
        "Poor Physical Health Days": "Avg physically unhealthy days",
    },
    title="PM2.5 exposure vs Physically Unhealthy Days",
)
fig1.show()


fig2 = px.scatter(
    newdf,
    x="new_pm2.5",
    y="Poor Mental Health Days",
    trendline="ols",
    labels={
        "pm25_composite": "PM2.5 composite (µg/m³)",
        "Poor Mental Health Days": "Avg mentally unhealthy days",
    },
    title="PM2.5 exposure vs Mentally Unhealthy Days",
)
fig2.show()

### Interpretation of Results & Limitations

The scatter plots of the composite PM₂.₅ index (`new_pm2.5`) versus county-level health outcomes (average physically unhealthy days and average mentally unhealthy days) do **not** show a clear positive relationship as hypothesized. In fact, in both charts the trend appears weak — and in some cases even suggests a negative slope. This null or ambiguous result leads to important caveats and reflections.

**Possible explanations and limitations:** 
- **Measurement and data quality issues.** PM₂.₅ data might not fully capture chronic exposure or acute spikes; the composite index is a simplification (weighted average). Health-days data may rely on survey or reporting methods and may not fully capture pollution-related health burden. Dropping rows with missing or invalid data (e.g. “ND”, “IN”, or NaN) may further bias the sample toward counties with more complete or reliable data.  
- **Cross-sectional nature and lack of temporal dimension.** The analysis is based on a cross-section at one point in time. Air pollution and health effects often accumulate over time (lag effects), which cannot be captured in a cross-section.  

**Conclusion (tentative):**  
The currently observed weak (or non-existent) correlation between PM₂.₅ exposure and health-day outcomes should be interpreted cautiously. It does *not* conclusively rule out health effects from air pollution — rather, it reflects limitations in the data, measurement, and design. Improved data (e.g. longitudinal exposure & health data, individual-level records), controlling for confounders, and more refined analysis would be needed to draw stronger conclusions.  

Nonetheless, reporting this “null / ambiguous” result is important — because it highlights the challenges of ecological, aggregated analyses and underscores the need for more robust, fine-grained data and careful methodological design in environmental health research.
